In [133]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import plotly.graph_objects as go
from trend_analisys_rsi_markov import TrendAnalyzer
import warnings
from scripts.assetsRoster import carteira_AC, carteira_HB, others
warnings.filterwarnings('ignore')


In [134]:
asset_ids = [
    'bitcoin', 'ethereum', 'binancecoin', 'ripple', 'cardano',
    'solana', 'polkadot', 'avalanche-2', 'chainlink', 'uniswap',
    'heyanon'
]

In [135]:
data_path='/Users/valter.rebelo/MissionControl/data/micro/candleData/'
btc_data_path='/Users/valter.rebelo/MissionControl/data/micro/candleData/bitcoin_candles.csv'
ssr_data_path='/Users/valter.rebelo/MissionControl/data/onchainData/BTC_SSR.csv'

In [136]:
analyzer = TrendAnalyzer(
    asset_ids=asset_ids,
    data_path=data_path,
    btc_data_path=btc_data_path,
    ssr_data_path=ssr_data_path,
    use_btc_adjusted=True,
    verbose=True,
    lookback_days=180,
    trend_metrics_lookback=365,
    markov_model_name='20250325_144631'
)

2025-03-27 16:16:13,348 - INFO - Loaded Markov volatility model: 20250325_144631
2025-03-27 16:16:13,354 - INFO - Successfully loaded SSR data with 2353 rows.


Data loaded successfully: 4102 records
Training set: 3281 records
Test set: 821 records
Model metadata:
  Saved on: 2025-03-25
  Data range: 2014-01-01 to 2025-03-24
  Records: 4101 total, 3075 train, 1026 test
Model loaded from /Users/valter.rebelo/MissionControl/models/markov_volatility_model_20250325_144631.pkl


In [ ]:

# Analyze assets
analyzer.analyze_multiple_assets()


Analyzing assets (static):  36%|███▋      | 4/11 [00:41<01:16, 10.98s/it]2025-03-27 16:16:55,660 - INFO - Successfully loaded data for cardano with 2716 rows.


In [ ]:
start_date='2024-01-1'
end_date='2025-03-25'

# Create a BTC-only trend-following portfolio
analyzer.create_portfolio(
    portfolio_name='btc_trend_following',
    #usd_conditions={'Short Term (USD)': ['Weak Bull','Strong Bull']},
    btc_only=True,
    btc_trend_gating=None,
    rsi_conditions_usd=True,
    use_ssr_signal=True,
    use_volatility_filter=True,
    volatility_weight=1
)

# Then create an altcoin portfolio that follows the Bitcoin portfolio
analyzer.create_portfolio(
    portfolio_name='big_wins_i',
    btc_conditions={'Short Term (BTC)': ['Strong Bull'], 'Overall (BTC)': ['Strong Bull']},
    usd_conditions={'Short Term (USD)': ['Strong Bull']},
    rsi_conditions_usd=True,
    rsi_conditions_btc=True,
    use_btc_rsi_signal=False,
    btc_only=False,  # This is an altcoin portfolio
    follow_portfolio='btc_trend_following',
    use_ssr_signal=False,
    use_ssr_gate=False,
    use_volatility_filter=True,
    volatility_weight=1
)

# Then create an altcoin portfolio that follows the Bitcoin portfolio
# Then create an altcoin portfolio that follows the Bitcoin portfolio
analyzer.create_portfolio(
    portfolio_name='big_wins_ii',
    btc_conditions={'Short Term (BTC)': ['Weak Bull','Strong Bull'], 'Overall (BTC)': ['Strong Bull']},
    usd_conditions={'Short Term (USD)': ['Strong Bull']},
    rsi_conditions_usd=True,
    rsi_conditions_btc=True,
    use_btc_rsi_signal=True,
    btc_only=False,  # This is an altcoin portfolio
    follow_portfolio='btc_trend_following',
    use_ssr_signal=False,
    use_ssr_gate=False,
    use_volatility_filter=True,
    volatility_weight=1
)


# Then create an altcoin portfolio that follows the Bitcoin portfolio
analyzer.create_portfolio(
    portfolio_name='fat_tails',
    btc_conditions={'Short Term (BTC)': ['Strong Bull'], 'Overall (BTC)': ['Weak Bull','Strong Bull']},
    usd_conditions={'Short Term (USD)': ['Strong Bull']},
    rsi_conditions_usd=True,
    rsi_conditions_btc=True,
    use_btc_rsi_signal=True,
    btc_only=False,  # This is an altcoin portfolio
    follow_portfolio='btc_trend_following',
    use_ssr_signal=False,
    use_ssr_gate=False,
    use_volatility_filter=True,
    volatility_weight=1
)

# Then create an altcoin portfolio that follows the Bitcoin portfolio
analyzer.create_portfolio(
    portfolio_name='experiment_i',
    btc_conditions={'Short Term (BTC)': ['Weak Bull','Strong Bull'], 'Overall (BTC)': ['Strong Bull']},
    usd_conditions={'Short Term (USD)': ['Strong Bear','Strong Bull']},
    rsi_conditions_usd=True,
    rsi_conditions_btc=True,
    use_btc_rsi_signal=False,
    btc_only=False,  # This is an altcoin portfolio
    follow_portfolio='btc_trend_following',
    use_ssr_signal=False,
    use_ssr_gate=False,
    use_volatility_filter=True,
    volatility_weight=1
)


########################################################

# Backtest the altcoin portfolio
results_alt = analyzer.backtest_portfolio(
    portfolio_name='big_wins_i',
    start_date=start_date,
    end_date=end_date,
    initial_capital=10000,
    alt_cost=0.005,
    signal_threshold=75
)

results_alt = analyzer.backtest_portfolio(
    portfolio_name='experiment_i',
    start_date=start_date,
    end_date=end_date,
    initial_capital=10000,
    alt_cost=0.005,
    signal_threshold=75
)

# Backtest the BTC trend-following portfolio
results_btc = analyzer.backtest_portfolio(
    portfolio_name='btc_trend_following',
    start_date=start_date,
    end_date=end_date,
    initial_capital=10000,
    alt_cost=0.001,
    signal_threshold=50
)

In [ ]:


# Plot performance for a specific date range
fig = analyzer.plot_individual_asset_performance(
    portfolio_name='experiment_i',
    btc_trend_portfolio_name='btc_trend_following',
    start_date=start_date,
    end_date=end_date,
    initial_capital=10000,
    show_plot=False,
   # asset_filter=['SOL']
)


In [ ]:
# Plot performance for a specific date range
fig = analyzer.plot_individual_asset_performance(
    portfolio_name='big_wins_i',
    btc_trend_portfolio_name='btc_trend_following',
    start_date=start_date,
    end_date=end_date,
    initial_capital=10000,
    show_plot=False,
   # asset_filter=['SOL']
)

In [ ]:
def get_signals_for_period(start_date, end_date, assets='all'):
    results_df = analyzer.get_portfolio_details(portfolio_name='altcoins_trend_following_btc').get('backtest_results').get('signals_df')
    
    # Filter by date range
    signals = results_df[(results_df.index >= start_date) & (results_df.index <= end_date)]
    
    # Filter by assets if specified
    if assets != 'all':
        if isinstance(assets, str):
            assets = [assets]
        signals = signals[signals['asset'].isin(assets)]
        
    return signals

# Example usage:
start_date = '2024-7-31'
end_date = '2025-3-25' 

assets = ['ETH']  # or 'all' for all assets
#assets = 'all'
signals = get_signals_for_period(start_date, end_date, assets)
signals[(signals['followed_portfolio_signal'] == 1) & (signals['final_decision'] == 0)]

In [ ]:
assets_held = analyzer.get_portfolio_details(portfolio_name='btc_gated_rsi_vol_momentum').get('backtest_results').get('results_df')
assets_held

import matplotlib.pyplot as plt

# Get the results dataframe

# Create the plot
plt.figure(figsize=(14, 6))

# Plot the assets held line
plt.plot(assets_held.index, assets_held['Assets_Held'], 'b-')

# Add vertical lines for each month
for date in assets_held.index[assets_held.index.is_month_start]:
    plt.axvline(x=date, color='gray', alpha=0.3, linestyle='--')

plt.title('Assets Held Over Time')
plt.ylabel('Number of Assets')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

In [ ]:
test_criteria = [{'Short Term (USD)': ['Strong Bull'],
                  'Short Term (BTC)': ['Strong Bull'],
                  'Overall (BTC)': ['Strong Bull']},
                  
                 {'Short Term (BTC)': ['Strong Bull'],
                  'Short Term (USD)': ['Weak Bull', 'Strong Bull']}]

In [ ]:
btc_data = analyzer.asset_data['maker'].get('classified_data')[['date', 'Short Term (USD)', 'Overall (USD)', 'RSI_Signal_close']]
btc_data[btc_data['date'] == data]


In [ ]:
results_df = analyzer.get_portfolio_details(portfolio_name='altcoins_trend_following_btc').get('backtest_results')
results_df

In [ ]:
signals_btc = analyzer.get_portfolio_details(portfolio_name='alt_short_momentum_low_vol').get('backtest_results').get('results_df')
signals_btc.head(60)

In [ ]:
analyzer.asset_data.get('maker')